# 3D CBCT Prognosis: Image + Tooth Metadata

Binary outcome:
- 0: Healed
- 1: Not-healed (Healing + Non-healed)

Inputs:
- 3D tooth-centered CBCT ROI
- tooth number from `Dataset A - Overview.xlsx`

The US/Universal tooth number is converted to:
- arch: maxillary vs mandibular
- tooth type: anterior vs premolar vs molar

The raw tooth number itself is not treated as a continuous predictor.

Automatic training mode:
- If `PRETRAINED_PATH` exists:
  - pretrained segmentation encoder LR = `1e-5`
  - new prognosis classifier LR = `1e-4`
- Otherwise:
  - whole network scratch LR = `1e-4`


In [2]:
# If needed:
# !pip install wandb scikit-learn scipy tqdm

from pathlib import Path
from collections import Counter
import random
import re

import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import wandb

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from prognosis_dataset_tooth_metadata import PrognosisDataset
from prognosis_model_tooth_metadata import PrognosisModel


SEED = 42
DATA_DIR = Path("../DSApre/roi_crop")
OUTCOME_XLSX = Path("Dataset A - Overview.xlsx")


NUM_CLASSES = 2
CLASS_NAMES = ["Healed", "Not-healed"]

USE_TOOTH_METADATA = True

# Model receives:
# [mandibular, anterior, premolar, molar]
TOOTH_METADATA_DIM = 4

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


seed_everything(SEED)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)


## 1. Discover ROI images and exclude oversized cases


In [17]:
dataset_all = PrognosisDataset(
    data_dir=DATA_DIR
)

excluded = [
    sample
    for sample in dataset_all.samples
    if np.any(
        np.asarray(sample["shape"])
        > np.array([176, 160, 288])

    )
]

print("Excluded oversized cases:", len(excluded))

for sample in excluded:
    print(
        sample["case_id"],
        tuple(sample["shape"]),
    )

dataset_all.samples = [
    sample
    for sample in dataset_all.samples
    if np.all(
        np.asarray(sample["shape"])
        <= np.array([176, 160, 288])

    )
]

dataset_all.target_shape = np.array([176, 160, 288]).copy()

print("\nRemaining image cases:", len(dataset_all.samples))
print("Target shape:", tuple(dataset_all.target_shape))


Excluded oversized cases: 7
DSA024pre (101, 89, 308)
DSA029pre (159, 136, 292)
DSA054pre (105, 115, 349)
DSA057pre (190, 137, 261)
DSA074pre (112, 122, 330)
DSA147pre (91, 78, 300)
DSA201pre (129, 165, 272)

Remaining image cases: 190
Target shape: (176, 160, 288)


## 2. Create prognosis labels and match them to available ROI images


In [18]:
df = pd.read_excel(
    OUTCOME_XLSX
)


def normalize_case_id(
    x,
):
    match = re.search(
        r"DSA[-_ ]?0*(\d+)",
        str(x),
        re.IGNORECASE,
    )

    if match is None:
        return None

    return (
        f"DSA"
        f"{int(match.group(1)):03d}"
    )


def extract_pai(
    x,
):
    match = re.search(
        r"\d+",
        str(x),
    )

    if match is None:
        return None

    return int(
        match.group()
    )


def assign_label(
    row,
):
    post_raw = str(
        row[
            "Follow-up CBCT-PAI [POST]"
        ]
    ).strip()

    if post_raw.lower() in {
        "-",
        "",
        "nan",
        "none",
    }:
        return None

    if (
        "extract"
        in post_raw.lower()
    ):
        return 2

    pre = extract_pai(
        row[
            "Pre-op CBCT-PAI [PRE]"
        ]
    )

    post = extract_pai(
        row[
            "Follow-up CBCT-PAI [POST]"
        ]
    )

    if post is None:
        return None

    # Healed
    if post <= 2:
        return 0

    if pre is None:
        return None

    # Healing
    if post < pre:
        return 1

    # Non-healed
    return 2


def normalize_column_name(
    x,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(x).lower(),
    )


def find_tooth_number_column(
    dataframe,
):
    """
    Find the tooth-number column without hard-coding one spelling.

    The selected column is printed so the mapping is auditable.
    """
    normalized = {
        normalize_column_name(col): col
        for col in dataframe.columns
    }

    exact_candidates = [
        "toothnumber",
        "toothno",
        "toothnum",
        "tooth",
        "toothid",
        "tooth#",
        "teethnumber",
    ]

    for candidate in exact_candidates:
        key = normalize_column_name(
            candidate
        )

        if key in normalized:
            return normalized[key]

    # Fallback for names such as
    # "Target Tooth Number" or "Interested Tooth #".
    for col in dataframe.columns:
        key = normalize_column_name(
            col
        )

        if (
            "tooth" in key
            and (
                "number" in key
                or "num" in key
                or "no" in key
            )
        ):
            return col

    raise KeyError(
        "Could not identify the tooth-number column in "
        "Dataset A - Overview.xlsx. Available columns:\\n"
        + "\\n".join(
            str(x)
            for x in dataframe.columns
        )
    )


TOOTH_NUMBER_COLUMN = find_tooth_number_column(
    df
)

print(
    "Using tooth-number column:",
    TOOTH_NUMBER_COLUMN,
)


df[
    "normalized_case_id"
] = (
    df["Sequence"]
    .apply(
        normalize_case_id
    )
)

df = df[
    df[
        "normalized_case_id"
    ].notna()
].copy()


df[
    "label"
] = df.apply(
    assign_label,
    axis=1,
)


# Convert tooth number to numeric.
df[
    "tooth_number"
] = pd.to_numeric(
    df[
        TOOTH_NUMBER_COLUMN
    ],
    errors="coerce",
)


df = df[
    df[
        "label"
    ].notna()
].copy()


df[
    "label"
] = (
    df["label"]
    .astype(int)
)


# Tooth number must follow US/Universal numbering.
valid_tooth = (
    df["tooth_number"]
    .between(
        1,
        32,
        inclusive="both",
    )
)

if USE_TOOTH_METADATA:
    n_invalid_tooth = int(
        (~valid_tooth).sum()
    )

    if n_invalid_tooth > 0:
        print(
            "Cases excluded because tooth number "
            "is missing/invalid:",
            n_invalid_tooth,
        )

    df = df[
        valid_tooth
    ].copy()


df[
    "tooth_number"
] = (
    df["tooth_number"]
    .astype(int)
)


label_map = dict(
    zip(
        df[
            "normalized_case_id"
        ],
        df[
            "label"
        ],
    )
)


tooth_number_map = dict(
    zip(
        df[
            "normalized_case_id"
        ],
        df[
            "tooth_number"
        ],
    )
)


prognosis_data = []


for sample in (
    dataset_all.samples
):
    normalized_id = (
        normalize_case_id(
            sample[
                "case_id"
            ]
        )
    )

    if (
        normalized_id
        not in label_map
    ):
        continue

    if (
        USE_TOOTH_METADATA
        and normalized_id
        not in tooth_number_map
    ):
        continue

    item = {
        "case_id":
            sample[
                "case_id"
            ],

        "image":
            str(
                sample[
                    "image"
                ]
            ),

        "label":
            int(
                label_map[
                    normalized_id
                ]
            ),
    }

    if USE_TOOTH_METADATA:
        item[
            "tooth_number"
        ] = int(
            tooth_number_map[
                normalized_id
            ]
        )

    prognosis_data.append(
        item
    )


print(
    "Usable cases:",
    len(
        prognosis_data
    ),
)


print(
    "Original classes:",
    Counter(
        x["label"]
        for x
        in prognosis_data
    ),
)


if USE_TOOTH_METADATA:
    tooth_table = pd.DataFrame(
        [
            {
                "case_id":
                    x["case_id"],

                "tooth_number":
                    x["tooth_number"],

                "arch":
                    PrognosisDataset
                    .tooth_metadata_from_number(
                        x["tooth_number"]
                    )[1],

                "tooth_type":
                    PrognosisDataset
                    .tooth_metadata_from_number(
                        x["tooth_number"]
                    )[2],
            }
            for x in prognosis_data
        ]
    )

    print(
        "\\nArch distribution:"
    )
    print(
        tooth_table[
            "arch"
        ].value_counts()
    )

    print(
        "\\nTooth type distribution:"
    )
    print(
        tooth_table[
            "tooth_type"
        ].value_counts()
    )


Using tooth-number column: Tooth Number [US]
Usable cases: 159
Original classes: Counter({0: 125, 2: 23, 1: 11})
\nArch distribution:
arch
Maxillary     83
Mandibular    76
Name: count, dtype: int64
\nTooth type distribution:
tooth_type
Molar       102
Premolar     51
Anterior      6
Name: count, dtype: int64


In [19]:
# PrognosisDataset discovery already prefers *_roi_img_refined.nii.gz
# when available, and dataset_all has already been filtered for oversized ROIs.
# Do not rebuild prognosis_data here because doing so would bypass that filtering.

print(
    "Using filtered prognosis_data:",
    len(prognosis_data),
    Counter(x["label"] for x in prognosis_data),
)


Using filtered prognosis_data: 159 Counter({0: 125, 2: 23, 1: 11})


## 3. Convert to the binary outcome and create the fixed train / validation / test split

The label conversion is performed **before** stratification:
- Healed → 0
- Healing or Non-healed → 1

The test set is held out and is not evaluated during training.


In [20]:
# Convert to binary before the split so stratification matches the actual task.
for item in prognosis_data:
    item["label"] = (
        0 if int(item["label"]) == 0 else 1
    )

print(
    "Binary class counts:",
    Counter(x["label"] for x in prognosis_data),
)

train_data, temp_data = train_test_split(
    prognosis_data,
    test_size=0.30,
    random_state=SEED,
    stratify=[
        x["label"]
        for x in prognosis_data
    ],
)

val_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=SEED,
    stratify=[
        x["label"]
        for x in temp_data
    ],
)


def print_split(name, data):
    print(
        f"{name}: {len(data)}",
        Counter(
            x["label"]
            for x in data
        ),
    )


print_split("Train", train_data)
print_split("Val", val_data)
print_split("Test", test_data)


# Save case IDs for reproducibility.
split_rows = []

for split_name, split_data in [
    ("train", train_data),
    ("val", val_data),
    ("test", test_data),
]:
    for x in split_data:
        split_rows.append(
            {
                "split": split_name,
                "case_id": x["case_id"],
                "label": x["label"],
                "image": x["image"],
                "tooth_number": x.get("tooth_number"),
            }
        )

Binary class counts: Counter({0: 125, 1: 34})
Train: 111 Counter({0: 87, 1: 24})
Val: 24 Counter({0: 19, 1: 5})
Test: 24 Counter({0: 19, 1: 5})


In [21]:
# Sanity check after the split.
assert set(x["label"] for x in train_data).issubset({0, 1})
assert set(x["label"] for x in val_data).issubset({0, 1})
assert set(x["label"] for x in test_data).issubset({0, 1})

print("Binary labels verified.")


Binary labels verified.


In [22]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    recall_score,
)


# ============================================================
# Tooth number -> metadata
# [mandibular, anterior, premolar, molar]
# ============================================================

ANTERIOR = {
    6, 7, 8, 9, 10, 11,
    22, 23, 24, 25, 26, 27
}

PREMOLAR = {
    4, 5, 12, 13,
    20, 21, 28, 29
}

MOLAR = {
    1, 2, 3,
    14, 15, 16,
    17, 18, 19,
    30, 31, 32
}


def tooth_to_metadata(tooth_number):
    tooth_number = int(tooth_number)

    mandibular = int(tooth_number >= 17)
    anterior = int(tooth_number in ANTERIOR)
    premolar = int(tooth_number in PREMOLAR)
    molar = int(tooth_number in MOLAR)

    return [
        mandibular,
        anterior,
        premolar,
        molar,
    ]


# ============================================================
# Build metadata-only datasets using EXACT SAME DL SPLIT
# ============================================================

def build_metadata_dataset(samples):

    X = np.array([
        tooth_to_metadata(sample["tooth_number"])
        for sample in samples
    ], dtype=np.float32)

    y = np.array([
        sample["label"]
        for sample in samples
    ], dtype=np.int64)

    case_ids = [
        sample["case_id"]
        for sample in samples
    ]

    return X, y, case_ids


X_train, y_train, train_ids = build_metadata_dataset(train_data)
X_val, y_val, val_ids = build_metadata_dataset(val_data)
X_test, y_test, test_ids = build_metadata_dataset(test_data)


print("Train:", X_train.shape, np.bincount(y_train))
print("Val:  ", X_val.shape, np.bincount(y_val))
print("Test: ", X_test.shape, np.bincount(y_test))


# ============================================================
# Logistic Regression
# ============================================================

model = LogisticRegression(
    class_weight="balanced",
    penalty="l2",
    C=1.0,
    solver="liblinear",
    random_state=42,
    max_iter=1000,
)

model.fit(
    X_train,
    y_train,
)


# ============================================================
# Evaluation
# ============================================================

def evaluate_metadata_model(model, X, y, name):

    prob = model.predict_proba(X)[:, 1]
    pred = model.predict(X)

    auc = roc_auc_score(
        y,
        prob,
    )

    bal_acc = balanced_accuracy_score(
        y,
        pred,
    )

    macro_f1 = f1_score(
        y,
        pred,
        average="macro",
        zero_division=0,
    )

    recalls = recall_score(
        y,
        pred,
        labels=[0, 1],
        average=None,
        zero_division=0,
    )

    cm = confusion_matrix(
        y,
        pred,
        labels=[0, 1],
    )

    print(f"\n{name}")
    print("=" * 40)
    print(f"AUC:               {auc:.4f}")
    print(f"Balanced accuracy: {bal_acc:.4f}")
    print(f"Macro F1:          {macro_f1:.4f}")
    print(f"Healed recall:     {recalls[0]:.4f}")
    print(f"Not-healed recall: {recalls[1]:.4f}")
    print("\nConfusion matrix:")
    print(cm)

    return prob


val_prob = evaluate_metadata_model(
    model,
    X_val,
    y_val,
    "Validation"
)

test_prob = evaluate_metadata_model(
    model,
    X_test,
    y_test,
    "Test"
)


# ============================================================
# Inspect learned coefficients
# ============================================================

feature_names = [
    "mandibular",
    "anterior",
    "premolar",
    "molar",
]

print("\nLogistic regression coefficients")
print("=" * 40)

for name, coef in zip(
    feature_names,
    model.coef_[0],
):
    print(f"{name:12s}: {coef:+.4f}")

Train: (111, 4) [87 24]
Val:   (24, 4) [19  5]
Test:  (24, 4) [19  5]

Validation
AUC:               0.7105
Balanced accuracy: 0.6632
Macro F1:          0.5556
Healed recall:     0.5263
Not-healed recall: 0.8000

Confusion matrix:
[[10  9]
 [ 1  4]]

Test
AUC:               0.5684
Balanced accuracy: 0.6579
Macro F1:          0.4574
Healed recall:     0.3158
Not-healed recall: 1.0000

Confusion matrix:
[[ 6 13]
 [ 0  5]]

Logistic regression coefficients
mandibular  : +0.0401
anterior    : +0.1044
premolar    : -0.2108
molar       : +0.0859


In [23]:
test_data

[{'case_id': 'DSA144pre',
  'image': '../DSApre/roi_crop/DSA144pre_roi_img_refined.nii.gz',
  'label': 0,
  'tooth_number': 30},
 {'case_id': 'DSA135pre',
  'image': '../DSApre/roi_crop/DSA135pre_roi_img_refined.nii.gz',
  'label': 0,
  'tooth_number': 4},
 {'case_id': 'DSA119pre',
  'image': '../DSApre/roi_crop/DSA119pre_roi_img_refined.nii.gz',
  'label': 0,
  'tooth_number': 15},
 {'case_id': 'DSA067pre',
  'image': '../DSApre/roi_crop/DSA067pre_roi_img_refined.nii.gz',
  'label': 1,
  'tooth_number': 15},
 {'case_id': 'DSA048pre',
  'image': '../DSApre/roi_crop/DSA048pre_roi_img_refined.nii.gz',
  'label': 0,
  'tooth_number': 10},
 {'case_id': 'DSA-108PRE',
  'image': '../DSApre/roi_crop/DSA-108PRE_roi_img_refined.nii.gz',
  'label': 0,
  'tooth_number': 31},
 {'case_id': 'DSA154pre',
  'image': '../DSApre/roi_crop/DSA154pre_roi_img_refined.nii.gz',
  'label': 1,
  'tooth_number': 3},
 {'case_id': 'DSA211pre',
  'image': '../DSApre/roi_crop/DSA211pre_roi_img_refined.nii.gz',
  'la

In [22]:
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    recall_score,
    confusion_matrix,
    roc_auc_score,
)


# ============================================================
# 1. SELECT THRESHOLD USING VALIDATION SET ONLY
# ============================================================

val_y_true = val_result["y_true"]
val_prob_not_healed = val_result["y_prob"][:, 1]

thresholds = np.linspace(
    0.20,
    0.80,
    50,
)

best_threshold = None
best_bal_acc = -np.inf

for threshold in thresholds:

    val_pred = (
        val_prob_not_healed
        >= threshold
    ).astype(int)

    bal_acc = balanced_accuracy_score(
        val_y_true,
        val_pred,
    )

    if bal_acc > best_bal_acc:
        best_bal_acc = bal_acc
        best_threshold = threshold


print(
    f"Validation-selected threshold: "
    f"{best_threshold:.3f}"
)

print(
    f"Validation balanced accuracy: "
    f"{best_bal_acc:.4f}"
)


# ============================================================
# 2. APPLY THE FIXED VALIDATION THRESHOLD TO TEST SET
# ============================================================

test_y_true = test_result["y_true"]
test_prob_not_healed = test_result["y_prob"][:, 1]

test_pred_thresholded = (
    test_prob_not_healed
    >= best_threshold
).astype(int)


# ============================================================
# 3. TEST METRICS
# ============================================================

test_auc = roc_auc_score(
    test_y_true,
    test_prob_not_healed,
)

test_bal_acc = balanced_accuracy_score(
    test_y_true,
    test_pred_thresholded,
)

test_macro_f1 = f1_score(
    test_y_true,
    test_pred_thresholded,
    average="macro",
    zero_division=0,
)

test_recall = recall_score(
    test_y_true,
    test_pred_thresholded,
    labels=[0, 1],
    average=None,
    zero_division=0,
)

test_cm = confusion_matrix(
    test_y_true,
    test_pred_thresholded,
    labels=[0, 1],
)


print(
    f"\nTest AUC: {test_auc:.4f}"
)

print(
    f"Test balanced accuracy: "
    f"{test_bal_acc:.4f}"
)

print(
    f"Test macro F1: "
    f"{test_macro_f1:.4f}"
)

print("\nPer-class recall:")
print(
    f"Healed: "
    f"{test_recall[0]:.4f}"
)
print(
    f"Not-healed: "
    f"{test_recall[1]:.4f}"
)

print(
    "\nConfusion matrix:"
)
print(
    test_cm
)


# ============================================================
# 4. SAVE THRESHOLDED TEST PREDICTIONS
# ============================================================

thresholded_test_df = pd.DataFrame(
    {
        "case_id":
            test_result["case_ids"],

        "label":
            test_y_true,

        "prob_healed":
            test_result["y_prob"][:, 0],

        "prob_not_healed":
            test_prob_not_healed,

        "prediction_0.5":
            (
                test_prob_not_healed
                >= 0.5
            ).astype(int),

        "prediction_val_threshold":
            test_pred_thresholded,
    }
)

print(
    thresholded_test_df
)

Validation-selected threshold: 0.347
Validation balanced accuracy: 0.8421

Test AUC: 0.7158
Test balanced accuracy: 0.5263
Test macro F1: 0.2286

Per-class recall:
Healed: 0.0526
Not-healed: 1.0000

Confusion matrix:
[[ 1 18]
 [ 0  5]]
       case_id  label  prob_healed  prob_not_healed  prediction_0.5  \
0    DSA144pre      0     0.579207         0.420793               0   
1    DSA135pre      0     0.581362         0.418638               0   
2    DSA119pre      0     0.652795         0.347205               0   
3    DSA067pre      1     0.533220         0.466780               0   
4    DSA048pre      0     0.617368         0.382632               0   
5   DSA-108PRE      0     0.559192         0.440808               0   
6    DSA154pre      1     0.553596         0.446404               0   
7    DSA211pre      0     0.571542         0.428458               0   
8    DSA036pre      1     0.549799         0.450201               0   
9    DSA199pre      0     0.607605         0.392395   